# State-dependent modulation of spiny projection neurons controlslevodopa-induced dyskinesia in a mouse model of Parkinson’s disease

**This Dataset (DANDI:001538):**
This dataset contains neurophysiology data from the Surmeier lab studying levodopa-induced dyskinesia (LID) in Parkinson's disease models. The data spans multiple experimental approaches:

- **Electrophysiology**: Patch-clamp recordings from striatal neurons (dSPNs and iSPNs)
- **Two-photon imaging**: Dendritic excitability and spine density measurements  
- **Behavioral analysis**: Abnormal involuntary movements (AIMs) and rotational behavior
- **Pharmacology**: Effects of various receptor agonists and antagonists
- **Optogenetics**: Selective stimulation of specific neuron populations

One of the nice features of dandi is that the data can be streamed directly anywhere from the world without download (although data can be loaded as well). To enable this streaming workflow we need to instantiate a dandi client.


In [1]:
import os

from dandi.dandiapi import DandiAPIClient
from dotenv import load_dotenv

# Load environment variables from .env file
# This is required at the moment because the data is on draft status
load_dotenv()
# Load token from environment variable
token = os.getenv("DANDI_API_TOKEN")
if not token:
    raise ValueError("DANDI_API_TOKEN environment variable not set. Please set it with your DANDI API token.")

dandiset_id = "001538"
client = DandiAPIClient(token=token)
client.authenticate(token=token)

dandiset = client.get_dandiset(dandiset_id, "draft")
assets = dandiset.get_assets()
assets_list = list(assets)

## Fetching data

Dandisets contain **Assets** which are individual data files within the dataset. For this project, each asset represents one NWB file containing:
- **One experimental session** from **one subject** (animal)
- All data streams recorded during that session (voltage, imaging, behavior)
- Complete experimental metadata and protocols
- Standardized data organization following NWB format

In general, assets can be fetched by an id or by the path. The path of an NWB file (asset) on DANDI is of the form:

`sub-<subject_id>/sub-<subject_id>_ses-<session_id>_[desc-<description>]_<modalities>.nwb`

Which includes the session_id. For this dataset, the session_id was encoded with the following pattern:

`<figure>++<measurement>++<cell_type>++<state>++<pharmacology>++<genotype>++<timestamp>`

For example, `"F3++SomExc++iSPN++OffState++none++WT++20160523154318"` indicates:
- Figure 3 data
- Somatic excitability measurement  
- Indirect pathway SPNs (iSPN)
- Off-state (no levodopa)
- No pharmacological treatment
- Wild-type genotype
- Recorded on May 23, 2016 at 15:43:18

To fetch data only for a certain figure or experimental condition, we will define the following set of utilities that will allow us to filter only the NWB files we are interested in.

In [ ]:
# Session ID Parsing Functions
# DANDI encodes asset paths as:
#   sub-<subject_id>/sub-<subject_id>_ses-<session_id>_<modalities>.nwb
# The reorganized dandiset uses a 5-token session_id pattern:
#   <cellType>++<state>++<pharm>++<genotype>++<date>
# Example:
#   'sub-Subject20190411LIDONtDTomato/..._ses-dSPN++OnState++D1RaSch++WT++20190411_icephys.nwb'
#   tokens = ['dSPN', 'OnState', 'D1RaSch', 'WT', '20190411']
# The MODALITY is in the filename suffix (icephys, ophys, image, behavior, icephys+ogen, ...).
# The FIGURE is identifiable from the subject_id (e.g., LIDOFF, LIDON, M1Rant, CDGIko, M1RCRSPR).


def get_session_id(asset_path: str) -> str:
    """Extract session_id (the part between _ses- and the next _) from a DANDI asset path."""
    if not asset_path:
        return ""
    fname = asset_path.split('/')[-1]
    if '_ses-' not in fname:
        return ""
    return fname.split('_ses-', 1)[1].rsplit('_', 1)[0]


def parse_session_id(session_id: str) -> dict:
    """Parse a 5-token session_id into named fields. Returns {} if unparseable."""
    tokens = session_id.split('++')
    if len(tokens) < 5:
        return {}
    return {
        'cell_type': tokens[0],   # dSPN | iSPN | pan (for pan-cellular biosensor / behavior)
        'state': tokens[1],       # OffState | OnState | CTRL | OFF | ON | PD
        'pharm': tokens[2],       # none | D1RaSch | D2Rasul | M1RaThp | ...
        'genotype': tokens[3],    # WT | CDGIKO | M1RCRISPR | iSPN-M1RKO | ...
        'date': tokens[4],        # YYYYMMDD
    }


def get_modality(asset_path: str) -> str:
    """Extract the trailing modality token from the filename (e.g., 'icephys', 'ophys', 'image', 'icephys+ogen')."""
    fname = asset_path.split('/')[-1]
    if not fname.endswith('.nwb'):
        return ''
    return fname[:-len('.nwb')].rsplit('_', 1)[-1]

## Patch-clamp data: per-mouse-day NWB files

Patch-clamp recordings (somatic + dendritic excitability) from Figures 1, 3, 6, 7, and 8 are organized as **one NWB file per mouse-day**. Each file contains all cells patched in that mouse on that day, with:

- One `CurrentClampSeries` per (cell, modality) holding all sweeps concatenated end-to-end (Pattern B). Individual sweeps are sliced via `idx_start + count` columns on the `IntracellularRecordingsTable`.
- The full 5-level icephys hierarchical chain (`IntracellularRecordings` -> `SimultaneousRecordings` -> `SequentialRecordings` -> `RepetitionsTable` -> `ExperimentalConditionsTable`).
- Paired line-scan kymographs for dendritic sweeps, consolidated per (cell, channel, dendrite location).
- A denormalized `RecordingsIndexTable` in `processing/recordings_index/` that pre-joins the chain for fast pandas-style querying.

The `session_id` follows a 5-token format: `<cellType>++<state>++<pharm>++<genotype>++<date>` (e.g., `dSPN++OffState++none++WT++20170202`). This identifies the experimental cohort the mouse-day belongs to.

In [ ]:
# Per-mouse-day patch-clamp files have modality suffix 'icephys' (not 'icephys+ogen').
# Each file holds all cells from one mouse on one experimental day.
patch_clamp_assets = [a for a in assets_list if get_modality(a.path) == 'icephys']
print(f'Per-mouse-day patch-clamp files: {len(patch_clamp_assets)}')

# Show breakdown by cell type and state, parsed from the session_id
from collections import Counter

breakdown = Counter()
for asset in patch_clamp_assets:
    fields = parse_session_id(get_session_id(asset.path))
    if fields:
        breakdown[(fields['cell_type'], fields['state'], fields['pharm'])] += 1
for (cell_type, state, pharm), n in sorted(breakdown.items()):
    print(f'  {cell_type:6s} {state:10s} pharm={pharm:10s}: {n} mouse-days')

### Stream one mouse-day file

We use `remfile` + `h5py` to stream the file from the DANDI S3 bucket directly, no download needed:

In [ ]:
import h5py
import remfile
from pynwb import NWBHDF5IO

# Pick any per-mouse-day file. We'll use an F1 dSPN paired-recording mouse-day
# (LID on-state with SCH 23390 paired-cell experiment).
asset = next(a for a in patch_clamp_assets if 'dSPN++OnState' in a.path and 'D1RaSch' in a.path)
print(f'Streaming: {asset.path}')

s3_url = asset.get_content_url(follow_redirects=1, strip_query=False)
file_system = remfile.File(s3_url)
h5_file = h5py.File(file_system, mode='r')
io = NWBHDF5IO(file=h5_file, load_namespaces=True)
nwbfile = io.read()

print(f'Subject: {nwbfile.subject.subject_id}')
print(f'Session start: {nwbfile.session_start_time}')
print(f'Cells in file (intracellular electrodes): {len(nwbfile.icephys_electrodes)}')
print(f'Acquisition objects: {len(nwbfile.acquisition)}')

### The RecordingsIndexTable: one-stop lookup

The cleanest entry point into a per-mouse-day file is the `RecordingsIndexTable` in `processing/recordings_index/`. It denormalizes the icephys chain into one flat table, one row per sweep, with both metadata columns and pointers back to the canonical chain:

In [ ]:
# The RecordingsIndexTable is a DynamicTable with one row per sweep.
idx_table = nwbfile.processing['recordings_index']['RecordingsIndexTable']
idx_df = idx_table.to_dataframe()

print(f'RecordingsIndexTable: {len(idx_df)} rows, {len(idx_df.columns)} columns')
print()
print('Columns:')
for col in idx_df.columns:
    print(f'  {col}')
print()
print('First 5 rows:')
idx_df.head()

In [ ]:
# Pandas-style filtering: get all somatic sweeps for Cell 1 under the SCH (D1R antagonist) condition
mask = (
    (idx_df['cell_id'] == 1)
    & (idx_df['condition_subfolder'] == 'LID on-state with SCH')
    & (idx_df['dendrite_type'] == 'Soma')
)
sub = idx_df[mask].sort_values('stimulus_current_pA')
print(f'Cell 1 / SCH / Soma: {len(sub)} sweeps')
print()
sub[['cell_id', 'condition_subfolder', 'dendrite_type', 'stimulus_current_pA', 'sweep_start_time_s']].head(10)

### Accessing individual sweep data (Pattern B slicing)

Each sweep's voltage trace lives inside a consolidated `CurrentClampSeries` that holds all sweeps from one cell concatenated. The `IntracellularRecordingsTable` row carries `idx_start` and `count` columns that slice into the consolidated array:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Grab the IRT row for one specific sweep
irt_df = nwbfile.intracellular_recordings.to_dataframe()
first_sch_sweep = irt_df[
    (irt_df[('intracellular_recordings', 'cell_id')] == 1)
    & (irt_df[('intracellular_recordings', 'condition_subfolder')] == 'LID on-state with SCH')
    & (irt_df[('intracellular_recordings', 'dendrite_type')] == 'Soma')
].iloc[0]

# The 'responses/response' column is a TimeSeriesReference: (idx_start, count, timeseries)
resp_ref = first_sch_sweep[('responses', 'response')]
ts = resp_ref.timeseries  # the consolidated CurrentClampSeries for this cell
idx_start = int(resp_ref.idx_start)
count = int(resp_ref.count)

print(f'CurrentClampSeries: {ts.name}')
print(f'  full series shape: {ts.data.shape}  (all sweeps concatenated)')
print(f'  this sweep slice: [{idx_start}:{idx_start + count}]  (count={count})')

# Read just the sweep we want (no need to load the whole series)
voltage_v = ts.data[idx_start : idx_start + count]
voltage_mv = voltage_v * 1000  # V -> mV
rate_hz = ts.rate
time_s = np.arange(count) / rate_hz

stim_pA = first_sch_sweep[('intracellular_recordings', 'stimulus_current_pA')]
plt.figure(figsize=(8, 3))
plt.plot(time_s, voltage_mv, lw=0.8)
plt.xlabel('Time (s)')
plt.ylabel('Voltage (mV)')
plt.title(f'Cell 1, post-SCH, {stim_pA:.0f} pA injection')
plt.tight_layout(); plt.show()

### Line-scan kymographs for dendritic sweeps

Dendritic sweeps include paired two-photon line-scan kymographs: Ch1 (Alexa Fluor 568 reference) and Ch2 (Fluo-4 calcium signal). These are stored as consolidated TimeSeries (one per cell × channel × dendrite location), and accessed via `TimeSeriesReferenceVectorData` columns on the RecordingsIndexTable:

In [ ]:
# Find a dendritic sweep with line-scan imaging
dend_mask = (idx_df['dendrite_type'] == 'Distal') & (idx_df['cell_id'] == 1)
dend_row = idx_df[dend_mask].iloc[0]

# line_scan_ch1 / line_scan_ch2 are TimeSeriesReferenceVectorData
ch1_ref = dend_row['line_scan_ch1']  # Alexa reference (R0)
ch2_ref = dend_row['line_scan_ch2']  # Fluo-4 (G)

ch1 = ch1_ref.timeseries.data[ch1_ref.idx_start : ch1_ref.idx_start + ch1_ref.count]
ch2 = ch2_ref.timeseries.data[ch2_ref.idx_start : ch2_ref.idx_start + ch2_ref.count]
print(f'Ch1 (Alexa, reference) shape: {ch1.shape}')
print(f'Ch2 (Fluo-4, calcium)  shape: {ch2.shape}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(ch1, aspect='auto', cmap='gray')
axes[0].set_title(f'Cell 1 Distal Ch1 (Alexa)\n{ch1_ref.timeseries.name}')
axes[0].set_xlabel('pixels along scan line'); axes[0].set_ylabel('line repetitions (time)')
axes[1].imshow(ch2, aspect='auto', cmap='hot')
axes[1].set_title(f'Cell 1 Distal Ch2 (Fluo-4)\n{ch2_ref.timeseries.name}')
axes[1].set_xlabel('pixels along scan line')
plt.tight_layout(); plt.show()

### The icephys hierarchical chain

For analysts who want to traverse the canonical chain (rather than use the denormalized index), all 5 levels are populated. Paired-recording cells (e.g., this mouse-day where Cell 1 was recorded baseline + post-SCH) get separate `ExperimentalConditions` rows:

In [ ]:
# ExperimentalConditions: top level, one row per condition the mouse was under
ec_df = nwbfile.icephys_experimental_conditions.to_dataframe()
print(f'ExperimentalConditions: {len(ec_df)} rows')
ec_df[['condition']]

## AIM behavioral data: DynamicTable

AIM (Abnormal Involuntary Movement) scores live in their own NWB files (one per scoring session). Each file contains a `DynamicTableAIMScore` in `processing/behavior/` with one row per timepoint (every 20 minutes post-L-DOPA):

In [ ]:
# AIM behavior files use 'pan++' cell-type prefix (whole-animal behavioral measurement)
# and have NO modality suffix (the file is named '..._ses-<session_id>.nwb' directly).
# We distinguish AIM from biosensor (_ophys) and videos (_image) by checking that
# the modality token starts with 'ses-' (which means no proper modality suffix was set).
aim_assets = [a for a in assets_list if '_ses-pan++' in a.path and get_modality(a.path).startswith('ses-')]
print(f'AIM behavior files: {len(aim_assets)}')

if aim_assets:
    asset = aim_assets[0]
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=False)
    fs = remfile.File(s3_url)
    h5_file = h5py.File(fs, mode='r')
    io_aim = NWBHDF5IO(file=h5_file, load_namespaces=True)
    nwb_aim = io_aim.read()
    aim_table = nwb_aim.processing['behavior']['DynamicTableAIMScore']
    aim_df = aim_table.to_dataframe()
    print(f'AIM scores: {len(aim_df)} timepoints')
    aim_df


## Optogenetic Experiments - oEPSC Analysis

### NWB Organization for Optogenetic Data

Optogenetic experiments combine voltage clamp electrophysiology with optical stimulation to study synaptic responses. These experiments measure optogenetically-evoked postsynaptic currents (oEPSCs) in striatal neurons following ChR2-mediated activation of cortical terminals.

**Data stored in `acquisition` module:**
- **VoltageClampSeries**: Raw current recordings from voltage clamp experiments
- **OptogeneticSeries**: Optical stimulation parameters and timing (stored in `stimulus` module)

**Data stored in `intervals` module:**
- **optogenetic_epochs_table**: Detailed timing information for stimulation and detection windows
- **Stage definitions**: pre_stimulation, stimulation, detection, post_detection, inter_sweep_interval

### Filtering for Optogenetic Data from Figure 2

We use the same filtering approach to identify optogenetic experiments (oEPSC measurements) from Figure 2. These files contain voltage clamp recordings with optical stimulation protocols.

In [ ]:
# Sr-oEPSC optogenetic recordings use modality suffix 'icephys+ogen'
# (icephys = patch-clamp voltage; ogen = optogenetic stimulation).
available_oepsc_assets = [a for a in assets_list if get_modality(a.path) == 'icephys+ogen']
print(f'Found {len(available_oepsc_assets)} Sr-oEPSC files:')
for i, asset in enumerate(available_oepsc_assets[:3]):
    session_id = get_session_id(asset.path)
    fields = parse_session_id(session_id)
    print(f'  {i+1}. {asset.path}')
    print(f'     Session: {session_id}')
    print(f'     Cell type: {fields.get("cell_type", "?")}, State: {fields.get("state", "?")}, Pharm: {fields.get("pharm", "?")}')

### Streaming an  NWB file with Optogenetic stimulation

Let's stream one optogenetic file to explore the voltage clamp data organization and optogenetic stimulation protocols.

In [ ]:
import h5py
import remfile
from pynwb import NWBHDF5IO

asset = available_oepsc_assets[0]
s3_url = asset.get_content_url(follow_redirects=1, strip_query=False)
file_system = remfile.File(s3_url)
file = h5py.File(file_system, mode="r")

io = NWBHDF5IO(file=file)
nwbfile = io.read()
nwbfile

In [ ]:
print("Available acquisition keys:")
acquisition_keys = list(nwbfile.acquisition.keys())
print(acquisition_keys)

print(f"\nFound {len([k for k in acquisition_keys if 'VoltageClamp' in k])} VoltageClampSeries in acquisition module")
print("These contain the raw current recordings from voltage clamp experiments")

### Optogenetic Epochs Table

The key to understanding optogenetic experiments is the `optogenetic_epochs_table` stored in the `intervals` module. This table provides precise timing information for each experimental phase within each trial, including stimulation and detection windows.

**Column explanations:**
- **start_time/stop_time**: Precise timing boundaries for each epoch (seconds)
- **stimulation_on**: Boolean indicating if optical stimulation is active
- **pulse_length_in_ms**: Duration of each optical pulse (when stimulation_on=True)
- **period_in_ms**: Inter-pulse interval for pulse trains
- **number_pulses_per_pulse_train**: Number of pulses in train (1 for single pulses)
- **number_trains**: Number of pulse trains
- **intertrain_interval_in_ms**: Time between trains
- **power_in_mW**: Optical power delivered
- **optical_fiber_locations**: Description of fiber placement
- **stage_name**: Experimental phase (pre_stimulation, stimulation, detection, post_detection, inter_sweep_interval)

In [ ]:
# Access the optogenetic epochs table
optogenetics_table_df = nwbfile.intervals["optogenetic_epochs_table"].to_dataframe()
optogenetics_table_df.head()

### Visualizing Voltage Clamp Responses with Stimulation Windows

Let's plot two voltage clamp sweeps side-by-side, showing both the stimulation and detection windows that are crucial for oEPSC analysis. The optogenetic epochs table provides precise timing information for each experimental phase.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get optogenetic timing information
optogenetics_table_df = nwbfile.intervals["optogenetic_epochs_table"].to_dataframe()
stimulation_entries_df = optogenetics_table_df[optogenetics_table_df["stimulation_on"] == True]
detection_entries_df = optogenetics_table_df[optogenetics_table_df["stage_name"] == "detection"]

# Create subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for trial_index, (trial_name, ax, title) in enumerate([(1, ax1, "VoltageClampSeriesSweep001"), (7, ax2, "VoltageClampSeriesSweep007")]):
    # Get voltage clamp response
    series_name = f"VoltageClampSeriesSweep{trial_index + 1:03d}"
    voltage_clamp_response = nwbfile.acquisition[series_name]
    timestamps_in_seconds = voltage_clamp_response.get_timestamps()
    data_in_amperes = voltage_clamp_response.get_data_in_units()
    data_in_pico_amperes = data_in_amperes * 1e12
    
    # Get timing info for this trial
    stimulation_info = stimulation_entries_df.iloc[trial_index]
    detection_info = detection_entries_df.iloc[trial_index]
    
    stimulation_start_ms = stimulation_info["start_time"] * 1000
    stimulation_stop_ms = stimulation_info["stop_time"] * 1000
    detection_start_ms = detection_info["start_time"] * 1000
    detection_stop_ms = detection_info["stop_time"] * 1000
    
    # Show data up to 50ms after detection window ends
    timestamps_in_milliseconds = timestamps_in_seconds * 1000
    starting_time = timestamps_in_milliseconds[0]
    end_time = detection_stop_ms + 50
    mask = (timestamps_in_milliseconds >= starting_time) & (timestamps_in_milliseconds <= end_time)
    timestamps_filtered = timestamps_in_milliseconds[mask]
    data_filtered = data_in_pico_amperes[mask]
    
    # Plot the data
    ax.plot(timestamps_filtered, data_filtered, 'b-', linewidth=1)
    ax.axvspan(stimulation_start_ms, stimulation_stop_ms, color="cyan", alpha=0.8, label="Stimulation")
    ax.axvspan(detection_start_ms, detection_stop_ms, color="gray", alpha=0.15, label="Detection")
    
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Current (pA)")
    ax.set_title(title)
    
    if trial_index == 0:  # Add legend only to first subplot
        ax.legend()

plt.tight_layout()
plt.show()

print("Stimulation and detection windows:")
print(f"Trial 1 - Stimulation: {stimulation_entries_df.iloc[0]['start_time']*1000:.1f}-{stimulation_entries_df.iloc[0]['stop_time']*1000:.1f} ms")
print(f"Trial 1 - Detection: {detection_entries_df.iloc[0]['start_time']*1000:.1f}-{detection_entries_df.iloc[0]['stop_time']*1000:.1f} ms")

### Event Detection and Analysis

#### MAD-Based Noise Estimation for Robust Event Detection

For accurate oEPSC event detection, we use **Median Absolute Deviation (MAD)** instead of standard deviation to estimate baseline noise. This approach is more robust to outliers (including the events themselves) and provides better threshold estimates.

**Key parameters:**
- **Detection window shift**: 100ms after original detection start (to avoid stimulation artifacts)
- **Event threshold**: ±5 × MAD-based standard deviation from median
- **Event merging**: Combine events within 1ms (handles multi-threshold crossings from single events)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Control parameters for event detection
detection_window_shift_ms = 100  # Shift detection window start by this many ms
event_merge_distance_ms = 1.0    # Merge events within this distance

def merge_nearby_events(event_times, event_amplitudes, merge_distance_ms):
    """Merge events within merge_distance_ms, keeping maximum amplitude"""
    if len(event_times) == 0:
        return event_times, event_amplitudes
    
    times = np.array(event_times)
    amplitudes = np.array(event_amplitudes)
    sorted_indices = np.argsort(times)
    times = times[sorted_indices]
    amplitudes = amplitudes[sorted_indices]
    
    merged_times = []
    merged_amplitudes = []
    
    i = 0
    while i < len(times):
        current_time = times[i]
        current_amp = amplitudes[i]
        
        # Find all events within merge_distance_ms
        j = i + 1
        max_amp = current_amp
        max_amp_time = current_time
        
        while j < len(times) and (times[j] - current_time) <= merge_distance_ms:
            if amplitudes[j] > max_amp:
                max_amp = amplitudes[j]
                max_amp_time = times[j]
            j += 1
        
        merged_times.append(max_amp_time)
        merged_amplitudes.append(max_amp)
        i = j
    
    return merged_times, merged_amplitudes

# Demonstrate event detection on one sweep
trial_index = 0
series_name = f"VoltageClampSeriesSweep{trial_index + 1:03d}"
voltage_clamp_response = nwbfile.acquisition[series_name]
timestamps_in_seconds = voltage_clamp_response.get_timestamps()
data_in_amperes = voltage_clamp_response.get_data_in_units()
data_in_pico_amperes = data_in_amperes * 1e12

# Get timing information
stimulation_info = stimulation_entries_df.iloc[trial_index]
detection_info = detection_entries_df.iloc[trial_index]

detection_start_ms_original = detection_info["start_time"] * 1000
detection_stop_ms = detection_info["stop_time"] * 1000
detection_start_ms_shifted = detection_start_ms_original + detection_window_shift_ms

# Filter data to shifted detection window
timestamps_in_milliseconds = timestamps_in_seconds * 1000
detection_mask = (timestamps_in_milliseconds >= detection_start_ms_shifted) & (timestamps_in_milliseconds <= detection_stop_ms)
detection_data = data_in_pico_amperes[detection_mask]
detection_timestamps = timestamps_in_milliseconds[detection_mask]

if len(detection_data) > 0:
    # Calculate MAD-based standard deviation
    noise_median = np.median(detection_data)
    mad = np.median(np.abs(detection_data - noise_median))
    mad_std = mad * 1.4826  # MAD to standard deviation conversion
    
    event_threshold_positive = noise_median + 5 * mad_std
    event_threshold_negative = noise_median - 5 * mad_std
    
    # Find events
    positive_event_indices = np.where(detection_data > event_threshold_positive)[0]
    negative_event_indices = np.where(detection_data < event_threshold_negative)[0]
    
    # Calculate amplitudes and merge nearby events
    positive_event_times_raw = detection_timestamps[positive_event_indices] if len(positive_event_indices) > 0 else []
    negative_event_times_raw = detection_timestamps[negative_event_indices] if len(negative_event_indices) > 0 else []
    
    positive_event_amplitudes_raw = detection_data[positive_event_indices] - noise_median if len(positive_event_indices) > 0 else []
    negative_event_amplitudes_raw = noise_median - detection_data[negative_event_indices] if len(negative_event_indices) > 0 else []
    
    positive_event_times_merged, positive_event_amplitudes_merged = merge_nearby_events(
        positive_event_times_raw, positive_event_amplitudes_raw, event_merge_distance_ms)
    negative_event_times_merged, negative_event_amplitudes_merged = merge_nearby_events(
        negative_event_times_raw, negative_event_amplitudes_raw, event_merge_distance_ms)
    
    print(f"Event Detection Results for {series_name}:")
    print(f"  Noise median: {noise_median:.2f} pA")
    print(f"  MAD-based std: {mad_std:.2f} pA")
    print(f"  Positive threshold: {event_threshold_positive:.2f} pA")
    print(f"  Negative threshold: {event_threshold_negative:.2f} pA")
    print(f"  Raw events: +{len(positive_event_amplitudes_raw)}, -{len(negative_event_amplitudes_raw)}")
    print(f"  Merged events: +{len(positive_event_amplitudes_merged)}, -{len(negative_event_amplitudes_merged)}")
    
    if len(positive_event_amplitudes_merged) > 0:
        print(f"  Positive event amplitudes: {[f'{amp:.1f}' for amp in positive_event_amplitudes_merged]} pA")
    if len(negative_event_amplitudes_merged) > 0:
        print(f"  Negative event amplitudes: {[f'{amp:.1f}' for amp in negative_event_amplitudes_merged]} pA")
        
else:
    print(f"No data in detection window for {series_name}")

### Comprehensive Event Analysis Grid

Finally, let's analyze all voltage clamp sweeps in this session systematically, showing event detection across all trials in a grid format. This provides a complete overview of optogenetic responses throughout the experimental session.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get all acquisition keys for voltage clamp series
acquisition_keys = [key for key in nwbfile.acquisition.keys() if key.startswith("VoltageClampSeries")]
num_sweeps = len(acquisition_keys)

# Calculate grid dimensions
cols = 3
rows = (num_sweeps + cols - 1) // cols

# Create subplots grid
fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
axes = axes.flatten() if num_sweeps > 1 else [axes]

# Initialize summary statistics
all_positive_amplitudes = []
all_negative_amplitudes = []
total_events = 0

# Process each sweep
for i, sweep_key in enumerate(acquisition_keys):
    trial_number = int(sweep_key.split('Sweep')[-1])
    
    # Get voltage clamp response
    voltage_clamp_response = nwbfile.acquisition[sweep_key]
    timestamps_in_seconds = voltage_clamp_response.get_timestamps()
    data_in_amperes = voltage_clamp_response.get_data_in_units()
    data_in_pico_amperes = data_in_amperes * 1e12
    
    # Get timing information for this trial
    if trial_number <= len(stimulation_entries_df) and trial_number <= len(detection_entries_df):
        stimulation_info = stimulation_entries_df.iloc[trial_number - 1]
        detection_info = detection_entries_df.iloc[trial_number - 1]
        
        stimulation_start_ms = stimulation_info["start_time"] * 1000
        stimulation_stop_ms = stimulation_info["stop_time"] * 1000
        detection_start_ms_original = detection_info["start_time"] * 1000
        detection_stop_ms = detection_info["stop_time"] * 1000
        detection_start_ms_shifted = detection_start_ms_original + detection_window_shift_ms
        
        # Show data up to 50ms after detection window ends
        timestamps_in_milliseconds = timestamps_in_seconds * 1000
        starting_time = timestamps_in_milliseconds[0]
        end_time = detection_stop_ms + 50
        mask = (timestamps_in_milliseconds >= starting_time) & (timestamps_in_milliseconds <= end_time)
        timestamps_filtered = timestamps_in_milliseconds[mask]
        data_filtered = data_in_pico_amperes[mask]
        
        # Calculate noise statistics and detect events
        detection_mask = (timestamps_filtered >= detection_start_ms_shifted) & (timestamps_filtered <= detection_stop_ms)
        detection_data = data_filtered[detection_mask]
        detection_timestamps = timestamps_filtered[detection_mask]
        
        if len(detection_data) > 0:
            # MAD-based noise estimation
            noise_median = np.median(detection_data)
            mad = np.median(np.abs(detection_data - noise_median))
            mad_std = mad * 1.4826
            
            event_threshold_positive = noise_median + 5 * mad_std
            event_threshold_negative = noise_median - 5 * mad_std
            
            # Find and merge events
            positive_event_indices = np.where(detection_data > event_threshold_positive)[0]
            negative_event_indices = np.where(detection_data < event_threshold_negative)[0]
            
            positive_event_times_raw = detection_timestamps[positive_event_indices] if len(positive_event_indices) > 0 else []
            negative_event_times_raw = detection_timestamps[negative_event_indices] if len(negative_event_indices) > 0 else []
            
            positive_event_amplitudes_raw = detection_data[positive_event_indices] - noise_median if len(positive_event_indices) > 0 else []
            negative_event_amplitudes_raw = noise_median - detection_data[negative_event_indices] if len(negative_event_indices) > 0 else []
            
            # Merge nearby events
            positive_event_times_merged, positive_event_amplitudes_merged = merge_nearby_events(
                positive_event_times_raw, positive_event_amplitudes_raw, event_merge_distance_ms)
            negative_event_times_merged, negative_event_amplitudes_merged = merge_nearby_events(
                negative_event_times_raw, negative_event_amplitudes_raw, event_merge_distance_ms)
            
            # Collect amplitudes for summary
            all_positive_amplitudes.extend(positive_event_amplitudes_merged)
            all_negative_amplitudes.extend(negative_event_amplitudes_merged)
            
            sweep_events = len(positive_event_amplitudes_merged) + len(negative_event_amplitudes_merged)
            total_events += sweep_events
            
            # Plot
            ax = axes[i]
            ax.plot(timestamps_filtered, data_filtered, 'b-', linewidth=1)
            ax.axvspan(stimulation_start_ms, stimulation_stop_ms, color="cyan", alpha=0.8, label="Stimulation")
            ax.axvspan(detection_start_ms_original, detection_stop_ms, color="gray", alpha=0.05, label="Original Detection")
            ax.axvspan(detection_start_ms_shifted, detection_stop_ms, color="lightblue", alpha=0.15, label="Shifted Detection")
            ax.axhline(y=noise_median, color='black', linestyle='-', alpha=0.5, linewidth=1)
            ax.axhline(y=event_threshold_positive, color='red', linestyle='--', alpha=0.7, linewidth=1)
            ax.axhline(y=event_threshold_negative, color='red', linestyle='--', alpha=0.7, linewidth=1)
            
            # Get plot limits for event markers
            y_min, y_max = ax.get_ylim()
            
            # Add event markers
            for event_time in positive_event_times_merged:
                ax.plot([event_time, event_time], [event_threshold_positive, y_max], 
                        color='red', alpha=0.8, linewidth=2)
            
            for event_time in negative_event_times_merged:
                ax.plot([event_time, event_time], [y_min, event_threshold_negative], 
                        color='red', alpha=0.8, linewidth=2)
            
            ax.set_xlabel("Time (ms)")
            ax.set_ylabel("Current (pA)")
            ax.set_title(f"{sweep_key} (Events: {sweep_events})")
            
            # Add legend only to first subplot
            if i == 0:
                ax.legend(loc='upper right', fontsize=8)

# Hide empty subplots
for i in range(num_sweeps, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

# Print comprehensive summary
print(f"\n=== COMPREHENSIVE oEPSC ANALYSIS SUMMARY ===")
print(f"Session: {get_session_id(asset.path)}")
print(f"Total sweeps analyzed: {num_sweeps}")
print(f"Total events detected: {total_events}")
print(f"Positive events: {len(all_positive_amplitudes)}")
print(f"Negative events: {len(all_negative_amplitudes)}")

if len(all_positive_amplitudes) > 0:
    print(f"\nPositive event statistics:")
    print(f"  Mean amplitude: {np.mean(all_positive_amplitudes):.2f} ± {np.std(all_positive_amplitudes):.2f} pA")
    print(f"  Median amplitude: {np.median(all_positive_amplitudes):.2f} pA")
    print(f"  Range: {np.min(all_positive_amplitudes):.1f} to {np.max(all_positive_amplitudes):.1f} pA")

if len(all_negative_amplitudes) > 0:
    print(f"\nNegative event statistics:")
    print(f"  Mean amplitude: {np.mean(all_negative_amplitudes):.2f} ± {np.std(all_negative_amplitudes):.2f} pA")
    print(f"  Median amplitude: {np.median(all_negative_amplitudes):.2f} pA")
    print(f"  Range: {np.min(all_negative_amplitudes):.1f} to {np.max(all_negative_amplitudes):.1f} pA")

print(f"\nAnalysis parameters:")
print(f"  Detection window shift: {detection_window_shift_ms} ms")
print(f"  Event merge distance: {event_merge_distance_ms} ms")
print(f"  Event threshold: ±5 x MAD-based standard deviation")

# Acetylcholine Biosensor Experiments - GRABACh3.0 Imaging

## NWB Organization for Biosensor Data

GRABACh3.0 acetylcholine biosensor experiments measure cholinergic signaling dynamics using genetically encoded fluorescent sensors. These experiments reveal state-dependent modulation of acetylcholine release in Parkinson's disease and levodopa-induced dyskinesia.

**Key experimental parameters:**
- **Biosensor**: GRABACh3.0 genetically encoded acetylcholine indicator
- **Imaging**: Two-photon microscopy at 920nm excitation, 520nm emission  
- **Target region**: Dorsal striatum
- **Stimulation**: Electrical stimulation to evoke acetylcholine release
- **Experimental conditions**: Control (CTRL), Parkinsonian (PD/6-OHDA), Dyskinetic off-state (OFF)

**Data stored in `processing/ophys` module:**
- **ROI Response Series**: Processed fluorescence traces from regions of interest containing biosensor signal
- **Plane Segmentation**: ROI definitions showing areas of biosensor expression
- **Trials Table**: Experimental metadata including stimulation parameters, treatments, and timing

**Analysis workflow:**
1. Extract fluorescence time series from ROI response series
2. Apply background subtraction and baseline normalization
3. Calculate area under curve (AUC) for stimulus-evoked responses
4. Normalize using calibration values (Fmax from acetylcholine, Fmin from TTX)
5. Compare responses across experimental conditions

### Filtering for Acetylcholine Biosensor Data from Figure 5

These experiments examine acetylcholine release dynamics across three experimental conditions that model different stages of Parkinson's disease pathology:
- **CTRL**: Control unlesioned mice with normal cholinergic signaling
- **PD**: Parkinsonian mice (6-OHDA lesioned, no levodopa treatment) showing reduced cholinergic tone
- **OFF**: Dyskinetic mice in off-state (24-48h after chronic levodopa) with altered cholinergic dynamics

In [ ]:
# Acetylcholine biosensor (GRABACh3.0) files use modality suffix 'ophys'.
# These are pan-cellular imaging — no patch-clamp, just optical signal.
available_oepsc_assets = [a for a in assets_list if get_modality(a.path) == 'ophys']
print(f'Found {len(available_oepsc_assets)} biosensor files:')
for i, asset in enumerate(available_oepsc_assets[:3]):
    session_id = get_session_id(asset.path)
    fields = parse_session_id(session_id)
    print(f'  {i+1}. {asset.path}')
    print(f'     Session: {session_id}')
    print(f'     State: {fields.get("state", "?")}, Date: {fields.get("date", "?")}')

### Streaming Acetylcholine Biosensor Data

The NWB files contain fluorescence time series from GRABACh3.0 biosensor experiments. Each file includes:
- **Trials table**: Metadata about stimulation timing, treatment conditions, and experimental parameters
- **ROI Response Series**: Processed fluorescence traces from regions of interest

In [ ]:
import h5py
import remfile
from pynwb import NWBHDF5IO

asset = available_oepsc_assets[0]
s3_url = asset.get_content_url(follow_redirects=1, strip_query=False)
file_system = remfile.File(s3_url)
file = h5py.File(file_system, mode="r")

io = NWBHDF5IO(file=file)
nwbfile = io.read()
nwbfile

We can extract the trials table as a data frame with the following method and inspect it.

In [ ]:
trials_df = nwbfile.trials.to_dataframe()
trials_df

And then we can see the structure of the session. Each trial is characterized by a treatment (control, quinpirole, sulpride, etc) a type of stimulation (single_pulse, burst or calibration) and also contains the name of the ROIResponseSeries that corresponds to this.

### Accessing Fluorescence Time Series Data


In [ ]:
fluorescence_module = nwbfile.processing["ophys"]["Fluorescence"]
fluorescence_module

In [ ]:
import matplotlib.pyplot as plt

trials_df = nwbfile.trials.to_dataframe()
trial_index = 10
trial_row = trials_df.iloc[trial_index]
stimulus_start_time = trial_row["stimulus_start_time"]
treatment = trial_row["treatment"]
roi_response_series_name = trial_row["roi_series_name"]
stimulation = trial_row["stimulation"]

roi_response_series = fluorescence_module[roi_response_series_name]


timestamps = roi_response_series.get_timestamps()
data = roi_response_series.data[:]

plt.plot(timestamps, data)
plt.axvline(stimulus_start_time, color='r', linestyle='--')
plt.title(f"{treatment} - {stimulation}")
plt.ylabel("Fluorescence (a.u.)")
plt.xlabel("Time (s)")

In [ ]:
import numpy as np

# pick 6 random trials that have a stimulus_start_time
valid_idx = trials_df[trials_df["stimulus_start_time"].notna()].index.values
rng = np.random.default_rng(42)
chosen = rng.choice(valid_idx, size=6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=False, sharey=True)
axes = axes.ravel()

# compute global y-limits across the chosen series (skip missing)
y_mins, y_maxs = [], []
for tid in chosen:
    row = trials_df.loc[tid]
    roi_name = row["roi_series_name"]
    series = fluorescence_module[roi_name]

    y = series.data[:]
    y_mins.append(np.nanmin(y))
    y_maxs.append(np.nanmax(y))

ymin = float(np.min(y_mins))
ymax = float(np.max(y_maxs))
pad = 0.05 * (ymax - ymin) if ymax > ymin else 1.0
ymin -= pad
ymax += pad

for ax, tid in zip(axes, chosen):
    row = trials_df.loc[tid]
    roi_name = row["roi_series_name"]
    stim_t = float(row["stimulus_start_time"])
    series = fluorescence_module[roi_name]

    ts = series.get_timestamps()
    y = series.data[:]
    rel_ts = ts - stim_t

    ax.plot(rel_ts, y, color="C0", lw=1)
    ax.axvline(0.0, color="r", linestyle="--", lw=1)
    ax.set_title(f"Observation {tid}: {row['treatment']} — {row['stimulation']}", fontsize=10)
    ax.set_xlabel("Time from stimulus (s)")
    ax.set_ylabel("Fluorescence (a.u.)")
    ax.set_ylim(ymin, ymax)

# overall layout
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()